In [0]:
FEATURE_TABLE_NAME = "workspace.marketing_campaign.gold_customer_features"
PREDICTION_TABLE_NAME = "workspace.marketing_campaign.customer_campaign_predictions"

features_df = spark.table(FEATURE_TABLE_NAME)

print(f"Feature table: {FEATURE_TABLE_NAME}")
print(f"Prediction table: {PREDICTION_TABLE_NAME}")
print(f"Rows: {features_df.count()}")

In [0]:
model_df = features_df.select(
    "customer_id",
    "education",
    "marital_status",
    "income",
    "customer_age",
    "customer_tenure_days",
    "has_children",
    "recency",
    "total_spend",
    "total_purchases",
    "numwebvisitsmonth",
    "acceptedcmp1",
    "acceptedcmp2",
    "acceptedcmp3",
    "acceptedcmp4",
    "acceptedcmp5",
    "accepted_previous_campaign",
    "complain",
    "response"
).dropna()

print(f"Model rows: {model_df.count()}")

In [0]:
train_df, scoring_df = model_df.randomSplit([0.8, 0.2], seed=42)

print(f"Training rows: {train_df.count()}")
print(f"Scoring rows: {scoring_df.count()}")

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression

categorical_columns = ["education", "marital_status"]

numeric_columns = [
    "income",
    "customer_age",
    "customer_tenure_days",
    "has_children",
    "recency",
    "total_spend",
    "total_purchases",
    "numwebvisitsmonth",
    "acceptedcmp1",
    "acceptedcmp2",
    "acceptedcmp3",
    "acceptedcmp4",
    "acceptedcmp5",
    "accepted_previous_campaign",
    "complain"
]

indexers = [
    StringIndexer(
        inputCol=column_name,
        outputCol=f"{column_name}_index",
        handleInvalid="keep"
    )
    for column_name in categorical_columns
]

encoders = [
    OneHotEncoder(
        inputCol=f"{column_name}_index",
        outputCol=f"{column_name}_encoded"
    )
    for column_name in categorical_columns
]

assembler = VectorAssembler(
    inputCols=numeric_columns + [f"{column_name}_encoded" for column_name in categorical_columns],
    outputCol="features"
)

logistic_regression = LogisticRegression(
    featuresCol="features",
    labelCol="response",
    predictionCol="prediction",
    probabilityCol="probability",
    maxIter=20
)

pipeline = Pipeline(stages=indexers + encoders + [assembler, logistic_regression])

model = pipeline.fit(train_df)

predictions_df = model.transform(scoring_df)

print("Predictions created.")

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

def get_response_probability(probability_vector):
    return float(probability_vector[1])

get_response_probability_udf = udf(get_response_probability, DoubleType())

predictions_with_probability_df = predictions_df.withColumn(
    "response_probability",
    get_response_probability_udf("probability")
)

In [0]:
from pyspark.sql.functions import col, current_timestamp

prediction_output_df = predictions_with_probability_df.select(
    "customer_id",
    "education",
    "marital_status",
    "income",
    "customer_age",
    "has_children",
    "total_spend",
    "total_purchases",
    "accepted_previous_campaign",
    col("response").alias("actual_response"),
    col("prediction").cast("int").alias("predicted_response"),
    "response_probability"
).withColumn(
    "prediction_timestamp",
    current_timestamp()
)

display(prediction_output_df.limit(10))

In [0]:
(
    prediction_output_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(PREDICTION_TABLE_NAME)
)

In [0]:
prediction_check_df = spark.table(PREDICTION_TABLE_NAME)

print(f"Rows saved: {prediction_check_df.count()}")
print(f"Columns saved: {len(prediction_check_df.columns)}")

display(
    prediction_check_df
    .orderBy("response_probability", ascending=False)
    .limit(20)
)